# Trabalho Prático 4 - Verificação de Segurança em Sistemas Híbridos

Este notebook implementa a verificação de segurança para o problema de tráfego marítimo descrito no enunciado do TP4.

O objetivo é garantir que dois navios, viajando em sentidos opostos num canal estreito, nunca ocupam o mesmo setor simultaneamente.

Utilizaremos:
- **Z3 Solver**: Para modelação lógica e verificação BMC (Bounded Model Checking).
- **Matplotlib**: Para visualização gráfica e animação do traço de execução (colisão ou seguro).

## 1. Configuração e Importações

Primeiro, importamos as bibliotecas necessárias. Certifique-se de que o ambiente tem `z3-solver` e `matplotlib` instalados.

In [6]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
from z3 import *
import sys

# Configuração para exibir animações no notebook
from matplotlib import rc
rc('animation', html='jshtml')

## 2. Modelação do Sistema em Z3

Definimos a física e a lógica do sistema.

### Parâmetros
- **Mapa**: 15 setores lineares ($s_0$ a $s_{14}$).
- **Navio 1**: Viaja de $0 \to 14$.
- **Navio 2**: Viaja de $14 \to 0$.
- **Física**: Discretização de Euler para velocidade e posição.
  - $v_{next} = v + \Delta t (F - \sigma v)$
  - $z_{next} = z + \Delta t \cdot v$

### Estado
Para cada passo $i$, o estado de cada navio é definido por:
- `s_i`: Índice do setor (Inteiro).
- `z_i`: Posição dentro do setor (Real, $0.0 \le z \le 1.0$).
- `v_i`: Velocidade (Real).

In [7]:
# Parâmetros Físicos Globais
SIGMA = 1.0   # Coeficiente de atrito
GAMMA = 2.0   # Força de aceleração
DT = 0.1      # Passo de tempo (delta t)
Z_MAX = 1.0   # Tamanho do setor (1 km)
NUM_SECTORS = 15

def declare_state(i):
    """Cria as variáveis Z3 para o estado no passo i."""
    state = {}
    # Navio 1
    state['s1'] = Int(f's1_{i}') 
    state['z1'] = Real(f'z1_{i}')
    state['v1'] = Real(f'v1_{i}')
    # Navio 2
    state['s2'] = Int(f's2_{i}')
    state['z2'] = Real(f'z2_{i}')
    state['v2'] = Real(f'v2_{i}')
    return state

def init(s):
    """Define o estado inicial (t=0)."""
    return And(
        s['s1'] == 0,  s['z1'] == 0.0, s['v1'] == 0.0, # Navio 1 no início
        s['s2'] == 14, s['z2'] == 0.0, s['v2'] == 0.0  # Navio 2 no fim
    )

## 3. Dinâmica e Transições

A função `trans` define como o sistema evolui de um estado `s` para o próximo `s_next`.

Inclui:
1.  **Física**: Atualização de velocidade e posição.
2.  **Lógica Discreta**: Mudança de setor quando $z \ge 1.0$.
    - Navio 1 incrementa setor.
    - Navio 2 decrementa setor.

In [8]:
def dynamics(v, force):
    """Equação de movimento discretizada."""
    return v + DT * (force - SIGMA * v)

def trans(s, s_next):
    """Define a transição de estado s -> s_next."""
    
    # --- Navio 1 (A -> B, s aumenta) ---
    force1 = GAMMA 
    v1_new = dynamics(s['v1'], force1)
    z1_new = s['z1'] + DT * s['v1']
    
    # Condição de mudança de setor: atingiu limite e não é o último setor
    crossing1 = And(s['z1'] >= Z_MAX, s['s1'] < NUM_SECTORS - 1)
    
    move1 = If(crossing1,
               And(s_next['s1'] == s['s1'] + 1,  # Avança setor
                   s_next['z1'] == 0.0,          # Reset z
                   s_next['v1'] == s['v1']),     # Mantém v
               And(s_next['s1'] == s['s1'],      # Mesmo setor
                   s_next['z1'] == z1_new,       # Atualiza z
                   s_next['v1'] == v1_new)       # Atualiza v
              )

    # --- Navio 2 (B -> A, s diminui) ---
    force2 = GAMMA
    v2_new = dynamics(s['v2'], force2)
    z2_new = s['z2'] + DT * s['v2']
    
    crossing2 = And(s['z2'] >= Z_MAX, s['s2'] > 0)
    
    move2 = If(crossing2,
               And(s_next['s2'] == s['s2'] - 1,  # Recua setor
                   s_next['z2'] == 0.0,
                   s_next['v2'] == s['v2']),
               And(s_next['s2'] == s['s2'],
                   s_next['z2'] == z2_new,
                   s_next['v2'] == v2_new)
              )

    return And(move1, move2)

## 4. Verificação BMC (Bounded Model Checking)

Procuramos um contra-exemplo para a propriedade de segurança.
**Propriedade de Segurança**: `s1 != s2` (Os navios não podem estar no mesmo setor).

O algoritmo BMC desenrola o sistema passo a passo ($k=1, 2, \dots$) e pergunta ao solver: *"Existe algum estado até ao passo $k$ onde a colisão ocorra?"*

In [9]:
def collision(s):
    """Define a condição de colisão (estado inseguro)."""
    return s['s1'] == s['s2']

def extract_trace(model, states, k_max):
    """Extrai os valores numéricos do modelo Z3."""
    trace = []
    for k in range(k_max + 1):
        # Helper para converter valor Z3 (racional) para float python
        def get_val(var):
            val_ref = model[var]
            if val_ref is None: return 0.0
            if is_int(val_ref): return float(val_ref.as_long())
            return float(val_ref.numerator_as_long()) / float(val_ref.denominator_as_long())

        trace.append({
            't': k * DT,
            's1': int(get_val(states[k]['s1'])), 
            'z1': get_val(states[k]['z1']),
            's2': int(get_val(states[k]['s2'])), 
            'z2': get_val(states[k]['z2'])
        })
    return trace

def run_bmc(limit=100):
    solver = Solver()
    states = [declare_state(0)]
    solver.add(init(states[0]))
    
    print(f"A iniciar BMC até k={limit}...")
    
    for k in range(1, limit + 1):
        # 1. Criar novo passo
        states.append(declare_state(k))
        solver.add(trans(states[k-1], states[k]))
        
        # 2. Verificar colisão
        solver.push()
        solver.add(collision(states[k]))
        
        if solver.check() == sat:
            print(f"!!! COLISÃO DETETADA NO PASSO {k} (Tempo: {k*DT:.1f}s) !!!")
            return extract_trace(solver.model(), states, k)
        
        solver.pop()
        
        if k % 10 == 0:
            print(f"Passo {k}: Seguro...")
            
    print("Nenhuma colisão encontrada.")
    return None

# Executar a verificação
trace_data = run_bmc(limit=100)

A iniciar BMC até k=100...
Passo 10: Seguro...
Passo 20: Seguro...
Passo 30: Seguro...
Passo 40: Seguro...
Passo 50: Seguro...
!!! COLISÃO DETETADA NO PASSO 57 (Tempo: 5.7s) !!!


## 5. Visualização e Animação

Se um traço de colisão for encontrado, geramos uma animação para visualizar o evento.
- **Navio 1 (Azul)**: Move-se da esquerda para a direita.
- **Navio 2 (Vermelho)**: Move-se da direita para a esquerda.
- **Colisão**: O setor fica vermelho quando ambos os navios lá estão.

In [10]:
def visualizar_traco(trace):
    if not trace:
        print("Sem dados para visualizar.")
        return None

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.set_xlim(-1, 16)
    ax.set_ylim(-1, 2)
    ax.set_aspect('equal')
    ax.set_title("Simulação de Tráfego Marítimo (Z3 Trace)")
    ax.set_xlabel("Setores (km)")
    ax.set_yticks([])
    
    # Desenhar setores estáticos
    sectors = []
    for i in range(15):
        rect = patches.Rectangle((i, 0), 1, 1, linewidth=1, edgecolor='black', facecolor='lightblue', alpha=0.3)
        ax.add_patch(rect)
        ax.text(i + 0.5, -0.3, f"S{i}", ha='center')
        sectors.append(rect)

    # Navios
    ship1_dot, = ax.plot([], [], 'bo', markersize=10, label='Navio 1 (A->B)')
    ship2_dot, = ax.plot([], [], 'ro', markersize=10, label='Navio 2 (B->A)')
    status_text = ax.text(0.5, 1.5, "", ha='center', fontsize=10, bbox=dict(facecolor='white', alpha=0.8))
    ax.legend(loc='upper right')

    def update(frame):
        data = trace[frame]
        
        # Posição visual = índice + z
        pos1 = data['s1'] + data['z1']
        pos2 = data['s2'] + data['z2']
        
        ship1_dot.set_data([pos1], [0.7])
        ship2_dot.set_data([pos2], [0.3])
        
        # Detetar colisão visualmente
        if data['s1'] == data['s2']:
            sectors[data['s1']].set_facecolor('red')
            status_text.set_color('red')
        else:
            for s in sectors: s.set_facecolor('lightblue')
            status_text.set_color('black')

        status_text.set_text(f"T={data['t']:.1f}s | N1: S{data['s1']} (z={data['z1']:.2f}) | N2: S{data['s2']} (z={data['z2']:.2f})")
        return ship1_dot, ship2_dot, status_text, *sectors

    ani = animation.FuncAnimation(fig, update, frames=len(trace), interval=100, blit=True)
    plt.close() # Evitar plot estático duplicado no notebook
    return ani

# Gerar a animação
visualizar_traco(trace_data)